Official universe: **51 companies**.
Phase B rebuilds are versioned under runs/phase_b/v1_51companies_20261114.
Keep all source passages; deduplicate only retrieval documents and retain provenance.
Recent means the last 90 days relative to the latest corpus publication.


In [ ]:
import pandas as pd
import numpy as np
import re
import hashlib
import time 


from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS


In [ ]:
df = pd.read_csv("../data/preprocessed_data/news_data_sentiment.csv")

print("Shape :", df.shape)
df.head()


In [ ]:
empty_text = (
    df["text"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
)

print("Textes vides :", empty_text.sum())

print(
    "Doublons textuels :",
    df.duplicated(subset=["text"]).sum()
)

duplicates = df[
    df.duplicated(
        subset=["company", "nid", "info_index"],
        keep=False
    )
]

print(
    "Doublons company/nid/info_index :",
    len(duplicates)
)


In [ ]:
rag_df = df.copy()

def clean_text(text):
    if not isinstance(text, str):
        return ""

    text = text.strip()
    text = re.sub(r"\s+", " ", text)

    return text

rag_df["text_clean"] = rag_df["text"].apply(clean_text)

rag_df = rag_df[
    rag_df["text_clean"] != ""
].copy()

print("Nombre de lignes :", len(rag_df))


In [ ]:
print(
    rag_df["n_tokens"]
    .sort_values(ascending=False)
    .head(10)
)


In [ ]:
# Preserve the complete news source for Forecast and all retrieval provenance.
# Token lengths are diagnostics; embedding/reranker tokenizer limits still apply.
print("Source passages over the legacy 300-token cutoff:", int(rag_df["n_tokens"].gt(300).sum()))


In [ ]:
rag_df["doc_id"] = (
    rag_df["ticker"].astype(str)
    + "_"
    + rag_df["nid"].astype(str)
    + "_"
    + rag_df["info_index"].astype(str)
)

def create_text_hash(text):
    return hashlib.sha1(
        text.encode("utf-8")
    ).hexdigest()

rag_df["text_hash"] = (
    rag_df["text_clean"]
    .apply(create_text_hash)
)

rag_df.head()


In [ ]:
rag_df.to_csv("../data/preprocessed_data/news_data_sentiment_rag.csv",index=False)
print("Dataset RAG sauvegardé.")

In [ ]:
from finagent.rag.corpus import deduplicate_rag_rows

# Retrieval representation only. rag_df and its CSV remain source-level rows
# for sentiment/forecasting. Every original doc_id and URL is kept in provenance.
retrieval_df = deduplicate_rag_rows(rag_df)
print("Source passages:", len(rag_df), "Retrieval documents:", len(retrieval_df))

def row_to_document(row):

    metadata = {
        "doc_id": str(row["doc_id"]),
        "text_hash": str(row["text_hash"]),
        "company": str(row["company"]),
        "ticker": str(row["ticker"]),
        "nid": str(row["nid"]),
        "info_index": int(row["info_index"]),
        "date": str(row["date"]),
        "article_title": str(row["article_title"]),
        "article_url": str(row["article_url"]),
        "sentiment": str(row["sentiment"]),
        "sentiment_score": float(row["sentiment_score"]),
        "positive_score": float(row["positive_score"]),
        "negative_score": float(row["negative_score"]),
        "neutral_score": float(row["neutral_score"]),
        "source": "The Edge Malaysia",
        "document_type": "news"
    }

    metadata["provenance"] = row["provenance"]
    metadata["original_doc_ids"] = row["original_doc_ids"]
    return Document(
        page_content=row["text_clean"],
        metadata=metadata
    )

In [ ]:
documents = [row_to_document(row) for _, row in retrieval_df.iterrows()]
print("Number of deduplicated retrieval documents:", len(documents))

In [ ]:
import pickle

with open("documents.pkl", "wb") as f:
    pickle.dump(documents, f)

print("Documents sauvegardés :", len(documents))

In [ ]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)


In [ ]:
test_embedding = embedding_model.embed_query(
    documents[0].page_content
)

print("Type :", type(test_embedding))
print("Dimension :", len(test_embedding))
print("Premières valeurs :", test_embedding[:10])


In [ ]:
vectorstore = FAISS.from_documents(
    documents=documents,
    embedding=embedding_model
)

print("Index FAISS créé.")


In [ ]:
from sentence_transformers import CrossEncoder
from finagent.rag.retrieval import retrieve_with_policy

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def retrieve_and_rerank(query, k=5, fetch_k=20):
    return retrieve_with_policy(
        query, k=k, fetch_k=fetch_k,
        corpus=documents, store=vectorstore, reranker=reranker,
    ).documents

In [ ]:
# ============================================================
# 8. TEST DU RETRIEVER FINAL
# ============================================================

question = """
What were Maybank's 4QFY2025 net profit
and dividend per share?
"""

retrieved_docs = retrieve_and_rerank(
    question,
    k=5
)


for i, doc in enumerate(
    retrieved_docs,
    start=1
):

    print("=" * 100)

    print(
        f"SOURCE {i}"
    )

    print(
        "Company :",
        doc.metadata["company"]
    )

    print(
        "Date :",
        doc.metadata["date"]
    )

    print(
        "URL :",
        doc.metadata["article_url"]
    )

    print()

    print(
        doc.page_content
    )

    print()

In [ ]:
# ============================================================
# 9. CONSTRUCTION DU CONTEXTE
# ============================================================

def build_context(documents):

    context_parts = []

    for i, doc in enumerate(
        documents,
        start=1
    ):

        metadata = doc.metadata

        context = f"""
[Source {i}]

Company: {metadata["company"]}
Ticker: {metadata["ticker"]}
Date: {metadata["date"]}
Source: {metadata["source"]}
URL: {metadata["article_url"]}

Content:
{doc.page_content}
"""

        context_parts.append(
            context.strip()
        )

    return "\n\n".join(
        context_parts
    )

In [ ]:
context = build_context(
    retrieved_docs
)

print(context)

In [ ]:
# ============================================================
# 10. PROMPT RAG
# ============================================================

from langchain_core.prompts import ChatPromptTemplate


rag_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are a financial research assistant specialized
in Malaysian listed companies.

Answer the user's question using ONLY the information
contained in the provided context.

Rules:

1. Do not use outside knowledge.

2. Do not invent financial figures, events,
   dates or explanations.

3. If the available context is insufficient,
   explicitly say that the retrieved documents
   do not contain enough information.

4. Cite the supporting source using:
   [Source 1], [Source 2], etc.

5. Be precise and concise.

6. When several documents provide relevant information,
   combine them carefully.

7. Do not treat information from one company as
   information about another company.
"""
        ),

        (
            "human",
            """
QUESTION:

{question}


CONTEXT:

{context}


ANSWER:
"""
        )
    ]
)

In [ ]:
# ============================================================
# 11. LLM
# ============================================================

import os
from getpass import getpass
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
load_dotenv()

api_key = os.getenv("GOOGLE_API_KEY")

# Demander la clé seulement si elle n'existe pas déjà
if not os.getenv("GOOGLE_API_KEY"):

    os.environ["GOOGLE_API_KEY"] = getpass(
        "Google API Key: "
    )


llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

In [ ]:
# ============================================================
# 12. GENERATION CHAIN
# ============================================================

from langchain_core.output_parsers import StrOutputParser


generation_chain = (
    rag_prompt
    | llm
    | StrOutputParser()
)

In [ ]:
def ask_rag(question, k=5):
    retrieval = retrieve_with_policy(
        question, k=k, fetch_k=max(20, k),
        corpus=documents, store=vectorstore, reranker=reranker,
    )
    if not retrieval.documents:
        answer = retrieval.notice
    else:
        context = build_context(retrieval.documents)
        answer = generation_chain.invoke({"question": question, "context": context})
        if retrieval.notice:
            answer = retrieval.notice + "\n\n" + answer
    return {"question": question, "answer": answer, "documents": retrieval.documents,
            "time_window": retrieval.time_window, "retrieval_notice": retrieval.notice}

In [ ]:
# ============================================================
# 14. FIRST RAG QUERY
# ============================================================

question = """
What were Maybank's 4QFY2025 net profit
and dividend per share?
"""


result = ask_rag(
    question
)


print(
    "QUESTION:"
)

print(
    result["question"]
)


print(
    "\nANSWER:"
)

print(
    result["answer"]
)

In [ ]:
# ============================================================
# 15. DISPLAY SOURCES
# ============================================================

def display_sources(documents):

    print(
        "\nSOURCES:"
    )

    for i, doc in enumerate(
        documents,
        start=1
    ):

        print(
            f"\n[Source {i}]"
        )

        print(
            "Company:",
            doc.metadata["company"]
        )

        print(
            "Date:",
            doc.metadata["date"]
        )

        print(
            "URL:",
            doc.metadata["article_url"]
        )

In [ ]:
# ============================================================
# 16. USER-FRIENDLY FUNCTION
# ============================================================

def ask_finagent(question):

    result = ask_rag(
        question
    )

    print(
        "\nQUESTION\n"
    )

    print(
        result["question"]
    )


    print(
        "\nANSWER\n"
    )

    print(
        result["answer"]
    )


    print(
        "\n" + "=" * 100
    )


    display_sources(
        result["documents"]
    )

In [ ]:
# ============================================================
# 17. GENERATION DES REPONSES DU BENCHMARK
# ============================================================

eval_df = pd.read_csv("../data/rag_evaluation_50_questions.csv")

test_df = eval_df

generated_results = []


for i, row in test_df.iterrows():

    question = row["question"]

    print(
        f"Question {i + 1}/{len(eval_df)}"
    )

    result = ask_rag(
        question
    )

    generated_results.append(
        {
            "question_id":
                row["question_id"],

            "question":
                question,

            "expected_answer":
                row["expected_answer"],

            "generated_answer":
                result["answer"],

            "retrieved_doc_ids":
                "|".join(
                    [
                        doc.metadata["doc_id"]
                        for doc in result["documents"]
                    ]
                )
        }
    )

    time.sleep(15)


generation_eval_df = pd.DataFrame(
    generated_results
)


display(
    generation_eval_df.head()
)

In [ ]:
generation_eval_df.to_csv("../data/rag_generation_results_50.csv", index=False)
